In [ ]:
# Load necessary libraries
install.packages(c("dplyr", "tidyr"))


In [1]:
library(dplyr)
library(tidyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
# Read the data; make sure the name in the file upload matches the name in the read_csv bubble
# Do not change the skeleton
data <- read.csv("Data.csv")
print(data)

    ID.Number Gender Age       Date Moon.Diameter..km. Moon.Distance..km.
1        1000 Female  19  5/22/2024               3476             395236
2        1001   Male  24  5/22/2024               3476             395236
3        1002   Male  19  5/22/2024               3476             395236
4        1003 Female  19  5/22/2024               3476             395236
5        1004 Female  18  5/22/2024               3476             395236
6        1005   Male  19  5/22/2024               3476             395236
7        1006 Female  18  5/22/2024               3476             395236
8        1007 Female  18  5/22/2024               3476             395236
9        1008 Female  19  5/22/2024               3476             395236
10       1009   Male  18  5/22/2024               3476             395236
11       1010 Female  18  5/22/2024               3476             395236
12       1011 Female  21  5/22/2024               3476             395236
13       1012   Male  17  7/20/2024   

In [6]:
# Function to calculate visual angle
calc_va_df <- function(df, size_col, dist_col) {
  size <- df[[size_col]]
  dist <- df[[dist_col]]
  (2 * atan(size / (2 * dist))) * (180 / pi)
}

# Calculate Visual Angle of All Objects
data <- data %>%
  mutate(
  # Visual Angles produced by probes
    Lower_Elevation_Perceptual_VA  = calc_va_df(data, "Round.1.Estimate.Template.Size..cm.", "Round.1.Estimate.Distance..cm."),
    Higher_Elevation_Perceptual_VA  = calc_va_df(data, "Round.2.Estimate.Template.Size..cm.", "Round.2.Estimate.Distance..cm."),
# Visual Angle of the Moon
    Real_Visual_Angle = calc_va_df(data, "Moon.Diameter..km.", "Moon.Distance..km."),
# Disparity Visual Angle
    Diameter = data$Moon.Diameter..km.,
    Distance = data$Moon.Distance..km.,
    Lower_Elevation = data$Elevation..deg.,
    Higher_Elevation = data$`Elevation..deg..1`)

data_long <- data %>%
  select(ID = 1, Gender, Date, Time, Age, Distance,
        Lower_Elevation_Perceptual_VA,
        Higher_Elevation_Perceptual_VA, Real_Visual_Angle,
        Lower_Elevation, Higher_Elevation) %>%
  pivot_longer(
    cols = c(Lower_Elevation_Perceptual_VA, Higher_Elevation_Perceptual_VA),
    names_to = "Measurement",
    values_to = "Reported_Visual_Angle") %>%
  filter(!is.na(Reported_Visual_Angle)) %>%
  mutate(
    Task = case_when(
      grepl("Perceptual", Measurement) ~ "Perceptual",
    ),
    Ratio_Visual_Angle = Reported_Visual_Angle / Real_Visual_Angle,

    Elevation = case_when(
      grepl("Lower_Elevation", Measurement) ~ Lower_Elevation,
      grepl("Higher_Elevation", Measurement) ~ Higher_Elevation
    ),
    Session = case_when(
      grepl("Lower", Measurement) ~ "Lower",
      grepl("Higher", Measurement) ~ "Higher"

    ),
    Template_Distance = case_when(
      grepl("Lower_Elevation", Measurement)  & grepl("Perceptual", Task) ~ data$Round.1.Estimate.Distance..cm.[match(ID, data$ID)],
      grepl("Higher_Elevation", Measurement)  & grepl("Perceptual", Task) ~ data$Round.2.Estimate.Distance..cm.[match(ID, data$ID)],
      TRUE ~ NA_real_
  ),
  Template_Size = case_when(
      grepl("Lower_Elevation", Measurement)  & grepl("Perceptual", Task) ~ data$Round.1.Estimate.Template.Size..cm.[match(ID, data$ID)],
      grepl("Higher_Elevation", Measurement)  & grepl("Perceptual", Task) ~ data$Round.2.Estimate.Template.Size..cm.[match(ID, data$ID)],
      TRUE ~ NA_real_
  )
  )

data_long <- data_long %>% select(-Higher_Elevation, -Lower_Elevation, -Measurement)
data_long <- filter(data_long)
write.csv(data_long, file = "FullMoonDataLong.csv", row.names = FALSE)